# LC 191 — Number of 1 Bits
**Day-56 | Bit Manipulation**

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> The trick <code>n &amp; (n-1)</code>
clears exactly the lowest set bit of n in one operation.
Counting how many times you can do this before n reaches 0 gives
you the Hamming weight — no need to inspect every bit position.
</div>

## Official Problem Statement

Given a positive integer `n`, write a function that returns the
number of set bits in its binary representation (also known as
the **Hamming weight**).

**Example 1:**
```
Input:  n = 11  (binary: 00000000000000000000000000001011)
Output: 3
```

**Example 2:**
```
Input:  n = 128  (binary: 00000000000000000000000010000000)
Output: 1
```

**Constraints:**
- `1 <= n <= 2^31 - 1`

## What This Is Actually Asking

Count the number of 1s in the 32-bit binary representation of n.

The naive way is to check each of the 32 bit positions by shifting
right and masking — always exactly 32 steps.

The smarter way uses `n & (n-1)`, which zeroes out the lowest set
bit, so we only iterate as many times as there are 1-bits.

If the number has few 1-bits (sparse), this is significantly
faster in practice, though both are O(1) for bounded 32-bit ints.

## Walk Through an Example by Hand

n = 13  (binary: 1101)

```
Step 1: n     = 1101  (13)
        n-1   = 1100  (12)
        n&(n-1)= 1100  (12)  <- cleared bit 0 (was 1)
        count = 1

Step 2: n     = 1100  (12)
        n-1   = 1011  (11)
        n&(n-1)= 1000  (8)   <- cleared bit 2 (was 1)
        count = 2

Step 3: n     = 1000  (8)
        n-1   = 0111  (7)
        n&(n-1)= 0000  (0)   <- cleared bit 3 (was 1)
        count = 3

Step 4: n = 0 -> stop.  Answer = 3
```

13 in binary is 1101 which has 3 set bits. Confirmed.

## The Picture

```
Why does n & (n-1) clear the lowest set bit?

  n   = ...1 0 0 0 0   <- lowest set bit at position k
  n-1 = ...0 1 1 1 1   <- bits below k flip to 1, bit k flips to 0
  AND = ...0 0 0 0 0   <- bit k and all below it become 0
  (bits ABOVE k are unchanged because borrow doesn't propagate)

  n     = 1 1 0 1   (13)
            ^---- lowest set bit is bit 0
  n-1   = 1 1 0 0   (12)
  n&n-1 = 1 1 0 0   lowest 1 cleared

  n     = 1 1 0 0   (12)
            ^---- lowest set bit is now bit 2
  n-1   = 1 0 1 1   (11)
  n&n-1 = 1 0 0 0   bit 2 cleared

  n     = 1 0 0 0   (8)
  n-1   = 0 1 1 1   (7)
  n&n-1 = 0 0 0 0   (0) -> done after 3 pops = 3 set bits
```

## When To Use This Pattern

- When you need to **count set bits**, think `n & (n-1)` loop.
- When you need to **check if a number is a power of 2**, think
  `n & (n-1) == 0` (exactly one set bit).
- When you need to **isolate the lowest set bit**, think `n & -n`
  (two's complement trick).
- When the problem mentions Hamming weight, distance, or parity,
  think bit manipulation.
- When iterating over all subsets of a bitmask, think `sub = (sub-1) & mask`.

## The Approach

Initialise a counter to 0. While n is not zero, execute
`n = n & (n-1)` to clear the lowest set bit and increment the
counter.

Each iteration removes exactly one 1-bit, so the loop runs
exactly as many times as there are set bits in n.

Return the counter. For 32-bit integers this is at most 32
iterations — effectively O(1).

In [1]:
# Imports (none required beyond builtins)
from typing import Callable

In [2]:
# --------------- Test Harness ---------------

def test_harness(func: Callable[[int], int]) -> None:
    """Run test cases for LC 191 and report PASSED/FAILED."""
    cases = [
        # (input, expected, label)
        (0b00000000000000000000000000001011,
         3, "LC example: 11"),
        (0b00000000000000000000000010000000,
         1, "LC example: 128"),
        (0b11111111111111111111111111111101,
         31, "LC example: 2^32-3 (31 bits set)"),
        (1, 1, "single bit: 1"),
        (0b11111111, 8, "all 8 lower bits set"),
        (2**31 - 1, 31, "max 31-bit value"),
    ]

    passed = 0
    for n, expected, label in cases:
        result = func(n)
        ok = result == expected
        status = "PASSED" if ok else "FAILED"
        if ok:
            passed += 1
        else:
            print(f"  {status} [{label}]")
            print(f"    got:      {result}")
            print(f"    expected: {expected}")
        print(f"  {status} [{label}]")

    total = len(cases)
    print(f"\n{'='*40}")
    print(f"Results: {passed}/{total} passed")
    if passed == total:
        print("All tests PASSED!")
    else:
        print(f"{total - passed} test(s) FAILED.")

In [3]:
def hammingWeight(n: int) -> int:
    """
    LC 191 — Number of 1 Bits (Hamming Weight)

    Strategy:
        n & (n-1) clears the lowest set bit.
        Count how many times until n == 0.

    Parameters
    ----------
    n : int — a positive 32-bit integer

    Returns
    -------
    int — number of 1 bits in binary representation of n

    Time : O(k) where k = number of set bits  (at most 32)
    Space: O(1)
    """
    res = 0
    while n:
        n &= (n-1)
        res+=1
    return res
    # Debug: print(f"hammingWeight({bin(n)})")
test_harness(hammingWeight)

  PASSED [LC example: 11]
  PASSED [LC example: 128]
  PASSED [LC example: 2^32-3 (31 bits set)]
  PASSED [single bit: 1]
  PASSED [all 8 lower bits set]
  PASSED [max 31-bit value]

Results: 6/6 passed
All tests PASSED!


In [ ]:
# Uncomment and run when solution is ready
# test_harness(hammingWeight)

## Complexity

| Approach | Time | Space |
|---|---|---|
| Brute force — check each of 32 bits | O(32) = O(1) | O(1) |
| n & (n-1) loop (optimal) | O(k), k = set bits | O(1) |
| Python built-in `bin(n).count('1')` | O(32) = O(1) | O(1) |

All approaches are O(1) for 32-bit integers since the bit width
is bounded. The `n & (n-1)` approach is the interviewer's
expected answer — it shows knowledge of the bit trick.

## Real World Connection

At Citi, Hamming weight is used in risk bitmask calculations:
counting how many risk flags are active for a portfolio in a
single bitwise operation rather than iterating over flags.

AWS networking uses set-bit counts to validate subnet masks and
count the number of available hosts in a CIDR block.

In data engineering, bitmaps (Roaring Bitmaps) store dense sets
of integers; cardinality queries use hardware `POPCNT` (population
count) which is exactly the Hamming weight of each 64-bit word.

The `n & (n-1)` trick also detects powers-of-2 in O(1), which is
used in hash table resizing and memory allocator block sizes.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra